### **Example of User-Based Collaborative Filtering for Recommender Systems**

In [18]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean, cityblock, cosine
from scipy.stats import pearsonr

##### User-based:
* assumes that similiar users give similiar ratings to each item.
##### Item-based:
* assumes that similiar items receive similiar ratings from each user

In [19]:
### a synthetic example dataset with user names and ratings for a handful of American holidays
### ratings range from 1 to 5; 1 being least favorite, 5 being most favorite holiday
df = pd.DataFrame({
    'Holiday': ['New Years', 'Easter', 'Independence Day', 'Halloween', 'Thanksgiving', 'Christmas'],
    'James': [3, 4, 5, 5, 2, 5],
    'Susan': [4, np.nan, 1, 1, 5, 5],
    'John': [3, 1, 5, 2, 5, 5],
    'Patricia': [3, 4, 5, np.nan, 5, 5],
    'Oliver': [np.nan, 1, 5, 1, 5, 4]
})
df

,Holiday,James,Susan,John,Patricia,Oliver
0,New Years,3,4.0,3,3.0,NaN
1,Easter,4,NaN,1,4.0,1.0
2,Independence Day,5,1.0,5,5.0,5.0
3,Halloween,5,1.0,2,NaN,1.0
4,Thanksgiving,2,5.0,5,5.0,5.0
5,Christmas,5,5.0,5,5.0,4.0


#### **Methods of Filtering Variables:**

In [20]:
### returning booleans from a chosen column
df['Holiday'] == 'Easter'

0    False
1     True
2    False
3    False
4    False
5    False
Name: Holiday, dtype: bool

In [21]:
### access a row and return a Series
df[df['Holiday'] == 'Easter']

,Holiday,James,Susan,John,Patricia,Oliver
1,Easter,4,NaN,1,4.0,1.0


In [22]:
### access a row and return an object
df.iloc[0]

Holiday     New Years
James               3
Susan             4.0
John                3
Patricia          3.0
Oliver            NaN
Name: 0, dtype: object

#### **Handling Missing Values:**

In [23]:
df['Susan'].notnull()

0     True
1    False
2     True
3     True
4     True
5     True
Name: Susan, dtype: bool

In [24]:
df['Patricia'].notnull() & df['James'].notnull()

0     True
1     True
2     True
3    False
4     True
5     True
dtype: bool

In [25]:
### select subset of rows/columns where the bool =True
df_notmissing = df[['Susan', 'Patricia']][df['Susan'].notnull() & df['Patricia'].notnull()]
df_notmissing

,Susan,Patricia
0,4.0,3.0
2,1.0,5.0
4,5.0,5.0
5,5.0,5.0


#### **Similiarity Metrics and Predicted Ratings:**
Different distance metrics to measure similarity:

#### 1. Euclidean:

In [26]:
sim_weights = {}
for user in df.columns[1:-1]:
    df_subset = df[['Oliver', user]][df['Oliver'].notnull() & df[user].notnull()]
    dist = euclidean(df_subset['Oliver'], df_subset[user])
    sim_weights[user] = 1.0 / (1.0 + dist)

sim_df = pd.DataFrame(
    sim_weights.items(),
    columns=['User', 'Similarity Weight']
)
print(sim_df)

       User  Similarity Weight
0     James           0.144591
1     Susan           0.195194
2      John           0.414214
3  Patricia           0.240253


Finding the predicted rating of 'Oliver' for 'New Years'. Get all ratings for a holiday by taking a slice of the dataframe; in this case we don't need the first column 'Holidays' or the column 'Oliver':

In [27]:
ratings = df.iloc[0][1:-1]
ratings

James         3
Susan       4.0
John          3
Patricia    3.0
Name: 0, dtype: object

Find the predicted rating by multiplying each user weight with its corresponding rating for the holiday New Years:

In [28]:
predicted_rating = 0.0
weights_sum = 0.0

for user in df.columns[1:-1]:
    predicted_rating += ratings[user] * sim_weights[user]
    weights_sum += sim_weights[user]

predicted_rating /= weights_sum
print('Predicted Rating: %f' % predicted_rating)

Predicted Rating: 3.196323


#### 2. Manhattan (Cityblock)

In [29]:
### repeat method, except using different distance metric
sim_weights = {}
for user in df.columns[1:-1]:
    df_subset = df[['Oliver', user]][df['Oliver'].notnull() & df[user].notnull()]
    dist = cityblock(df_subset['Oliver'], df_subset[user])
    sim_weights[user] = 1.0 / (1.0 + dist)

sim_df = pd.DataFrame(
    sim_weights.items(),
    columns=['User', 'Similarity Weight']
)
print(sim_df)

       User  Similarity Weight
0     James           0.083333
1     Susan           0.166667
2      John           0.333333
3  Patricia           0.200000


In [30]:
predicted_rating = 0.0
weights_sum = 0.0

for user in df.columns[1:-1]:
    predicted_rating += ratings[user] * sim_weights[user]
    weights_sum += sim_weights[user]
    
predicted_rating /= weights_sum
print('Predicted Rating: %f' % predicted_rating)

Predicted Rating: 3.212766


#### 3. Cosine

In [31]:
sim_weights = {}
for user in df.columns[1:-1]:
    df_subset = df[['Oliver', user]][df['Oliver'].notnull() & df[user].notnull()]
    dist = cosine(df_subset['Oliver'], df_subset[user])
    sim_weights[user] = 1.0 / (1.0 + dist)

sim_df = pd.DataFrame(
    sim_weights.items(),
    columns=['User', 'Similarity Weight']
)
print(sim_df)

       User  Similarity Weight
0     James           0.830755
1     Susan           0.880308
2      John           0.989849
3  Patricia           0.950305


In [32]:
predicted_rating = 0.0
weights_sum = 0.0

for user in df.columns[1:-1]:
    predicted_rating += ratings[user] * sim_weights[user]
    weights_sum += sim_weights[user]
    
predicted_rating /= weights_sum
print('Predicted Rating: %f' % predicted_rating)

Predicted Rating: 3.241100


#### 4. Pearson Correlation Coefficient

In [33]:
sim_weights = {}
for user in df.columns[1:-1]:
    df_subset = df[['Oliver', user]][df['Oliver'].notnull() & df[user].notnull()]
    sim_weights[user] = pearsonr(df_subset['Oliver'], df_subset[user])[0]
    
sim_df = pd.DataFrame(
    sim_weights.items(),
    columns=['User', 'Similarity Weight']
)
print(sim_df)

       User  Similarity Weight
0     James          -0.299392
1     Susan           0.457496
2      John           0.963705
3  Patricia           0.968496


In [34]:
predicted_rating = 0.0
weights_sum = 0.0

ratings = df.iloc[0][1:-1]
for user in df.columns[1:-1]:
    if (not np.isnan(sim_weights[user])):
        predicted_rating += ratings[user] * sim_weights[user]
        weights_sum += sim_weights[user]
    
predicted_rating /= weights_sum
print('Predicted Rating: %f' % predicted_rating)

Predicted Rating: 3.218866
